In [29]:
import os
import sys
import torch

In [30]:
gpu_ready = torch.cuda.is_available()
print(f"CUDA available: {gpu_ready}")
if gpu_ready:
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("No CUDA GPU is visible in this session.")


CUDA available: True
GPU device: NVIDIA H100 NVL
CUDA version: 12.4


In [31]:
repo_dir = "/data/UG/Kiranmoy/Repo_Destination/High-Level-Military-Camouflage-Detection/GroundingDINO"
vendor_dir = "/data/UG/Kiranmoy/Repo_Destination/High-Level-Military-Camouflage-Detection/.groundingdino_pkgs"

cache_dir = os.path.join(vendor_dir, ".hf_cache")
os.makedirs(cache_dir, exist_ok=True)
os.environ.setdefault("HF_HOME", cache_dir)
os.environ.setdefault("HF_HUB_CACHE", os.path.join(cache_dir, "hub"))
os.environ.setdefault("MPLCONFIGDIR", os.path.join(vendor_dir, ".mplconfig"))
for path in (vendor_dir, repo_dir):
    if path not in sys.path:
        sys.path.insert(0, path)
os.chdir(repo_dir)


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using GroundingDINO from {repo_dir} with local dependencies in {vendor_dir}")
print(f"CUDA available: {torch.cuda.is_available()} | selected device: {DEVICE}")

Using GroundingDINO from /data/UG/Kiranmoy/Repo_Destination/High-Level-Military-Camouflage-Detection/GroundingDINO with local dependencies in /data/UG/Kiranmoy/Repo_Destination/High-Level-Military-Camouflage-Detection/.groundingdino_pkgs
CUDA available: True | selected device: cuda


In [12]:
!mkdir -p weights

!wget -O weights/groundingdino_swint_ogc.pth \
https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth

--2026-06-10 12:46:48--  https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth
Resolving github.com (github.com)... 20.207.73.82
Connecting to github.com (github.com)|20.207.73.82|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/611591640/f221e500-c2fc-4fd3-b84e-8ad92a6923f3?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-06-10T08%3A03%3A05Z&rscd=attachment%3B+filename%3Dgroundingdino_swint_ogc.pth&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-06-10T07%3A02%3A14Z&ske=2026-06-10T08%3A03%3A05Z&sks=b&skv=2018-11-09&sig=GpaiGWafYZRsfwfAdJWZcChl1hGoP5LyZxjZemw85PY%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc4MTA3OTQwOCwibmJmIjoxNzgxMDc1ODA4LCJwYXRo

In [32]:
import os
import json
import cv2
import torch
import numpy as np
from tqdm import tqdm
from pathlib import Path

from groundingdino.util.inference import load_model
from groundingdino.util.inference import load_image
from groundingdino.util.inference import predict

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

In [33]:
from pathlib import Path

DATASET_ROOT = Path(
    "/data/UG/Kiranmoy/datasets"
)

COCO_GT = (
    DATASET_ROOT
    / "MHCD2022_COCO_Labels"
    / "test_coco.json"
)

IMAGE_DIR = (
    DATASET_ROOT
    / "Military-Camouflage-MHCD2022"
    / "JPEGImages"
)

CONFIG_PATH = (
    "groundingdino/config/"
    "GroundingDINO_SwinT_OGC.py"
)

WEIGHTS_PATH = (
    "weights/"
    "groundingdino_swint_ogc.pth"
)

In [36]:
from pycocotools.coco import COCO

coco_gt = COCO(str(COCO_GT))

cats = coco_gt.loadCats(coco_gt.getCatIds())

cats

loading annotations into memory...
Done (t=0.01s)
creating index...
index created!


[{'id': 0, 'name': 'person'},
 {'id': 1, 'name': 'military vehicle'},
 {'id': 2, 'name': 'tank'},
 {'id': 3, 'name': 'aeroplane'},
 {'id': 4, 'name': 'warship'}]

In [37]:
DEVICE = "cuda"

from transformers.tokenization_utils_base import AddedToken

# Compatibility shim for the vendored transformers build in this repo.
# Some environments ship an AddedToken implementation without a `.content`
# attribute, while GroundingDINO expects it during tokenizer loading.
if not hasattr(AddedToken, "content"):
    AddedToken.content = property(lambda self: str(self))

model = load_model(
    CONFIG_PATH,
    WEIGHTS_PATH
)

print("Model loaded successfully")


final text_encoder_type: bert-base-uncased


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Model loaded successfully


In [43]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using:", device)

model = model.to(device)

Using: cuda


In [44]:
print(next(model.parameters()).device)

cuda:0
